In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class StowageHighLevelEnv(gym.Env):
    """
    Gymnasium wrapper for high-level agent (bay selection)
    Wraps your existing stowage environment for SB3 compatibility
    """
    metadata = {"render_modes": ["human"]}

    def __init__(self, stowage_env):
        """
        Args:
            stowage_env: Your existing stowage environment instance
        """
        super().__init__()
        self.env = stowage_env
        self.low_level_agent = None  # Will be set if needed
        
        # Action space: select one of the yard bays
        # Adjust based on your actual number of bays
        self.num_bays = stowage_env.yard_shape[0]//2 + stowage_env.yard_shape[0]
        self.action_space = spaces.Discrete(self.num_bays)
        
        # Observation space: your observation vector
        # Adjust size based on your actual observation
        num_vessel_bay = stowage_env.vessel_shape[0]//2 + stowage_env.vessel_shape[0]
        num_yard_bay = stowage_env.yard_shape[0]//2 + stowage_env.yard_shape[0]
        num_slot_attrs = 5  # Adjust to your num_slot_attrs
        
        total_vessel_coords = num_vessel_bay * stowage_env.vessel_shape[1] * stowage_env.vessel_shape[2]
        total_yard_coords = num_yard_bay * stowage_env.yard_shape[1] * stowage_env.yard_shape[2]
        
        obs_size = (total_vessel_coords + total_yard_coords) * num_slot_attrs + num_slot_attrs
        
        self.observation_space = spaces.Box(
            low=0, 
            high=255, 
            shape=(obs_size,), 
            dtype=np.float32
        )
        
        self.current_observation = None
        self.valid_actions = None

    def reset(self, seed=None, options=None):
        """Reset environment and get initial observation"""
        super().reset(seed=seed)
        
        # Reset your stowage environment
        self.current_observation = self.env.reset()
        
        # Get valid actions from your environment
        self.valid_actions = self.env.get_valid_actions()
        
        return self.current_observation.astype(np.float32) / 255.0, {}

    def step(self, action):
        """
        Execute one step in the environment
        action: bay selected by PPO agent
        """
        # Validate action
        if len(self.valid_actions) == 0:
            return self.current_observation.astype(np.float32) / 255.0, 0, True, False, {}
        
        # Get yard state to extract valid bays
        yard_state = self._parse_yard_state(self.current_observation)
        valid_bays = set(yard_state[self.valid_actions, 0].astype(int))  # StateIds.BAY = 0
        
        # If action not in valid bays, use random valid bay
        if int(action) not in valid_bays:
            action = int(np.random.choice(list(valid_bays)))
        
        selected_bay = int(action)
        
        # If you have a low-level agent, use it to select the slot
        if self.low_level_agent is not None:
            selected_slot = self.low_level_agent.get_action(
                self.current_observation, 
                self.valid_actions, 
                selected_bay
            )
        else:
            # Or just select random slot in bay as fallback
            bay_mask = yard_state[self.valid_actions, 0] == selected_bay
            bay_valid_actions = np.array(self.valid_actions)[bay_mask]
            selected_slot = int(np.random.choice(bay_valid_actions))
        
        # Execute action in your stowage environment
        self.current_observation, reward, done, info = self.env.step(selected_slot)
        
        # Get new valid actions
        self.valid_actions = self.env.get_valid_actions()
        
        # Normalize observation
        obs = self.current_observation.astype(np.float32) / 255.0
        
        return obs, float(reward), done, False, info
    
    def _parse_yard_state(self, observation):
        """Parse yard state from observation"""
        observation = observation.astype(int)
        num_vessel_bay = self.env.vessel_shape[0]//2 + self.env.vessel_shape[0]
        num_yard_bay = self.env.yard_shape[0]//2 + self.env.yard_shape[0]
        num_slot_attrs = 5  # Adjust
        
        total_vessel_coords = num_vessel_bay * self.env.vessel_shape[1] * self.env.vessel_shape[2]
        total_yard_coords = num_yard_bay * self.env.yard_shape[1] * self.env.yard_shape[2]
        
        vessel_end = total_vessel_coords * num_slot_attrs
        yard_end = vessel_end + (total_yard_coords * num_slot_attrs)
        
        yard_state = observation[vessel_end:yard_end].reshape(total_yard_coords, num_slot_attrs)
        return yard_state

    def render(self):
        pass

In [ ]:
# Add this to your imports
import os
from stable_baselines3 import PPO
from gymnasium_wrapper import StowageHighLevelEnv

# ...existing code...

class HighLevelAgent(BaseAgent):
    """
    High-Level Agent - now supports SB3 PPO
    """

    def __init__(self, vessel_shape, yard_shape, num_slot_attrs, policy_type="rule_based", 
                 stowage_env=None, rl_model_path=None):
        super().__init__(AgentLevel.HIGH_LEVEL)

        self.vessel_shape = vessel_shape
        self.yard_shape = yard_shape
        self.policy_type = policy_type
        self.num_slot_attrs = num_slot_attrs
        self.episode_history = []

        self.num_vessel_bay = vessel_shape[0]//2 + vessel_shape[0]
        self.num_yard_bay = yard_shape[0]//2 + yard_shape[0]

        self.total_vessel_coords = self.num_vessel_bay * vessel_shape[1] * vessel_shape[2]
        self.total_yard_coords = self.num_yard_bay * yard_shape[1] * yard_shape[2]

        self.rl_model = None
        
        # Initialize RL model if needed
        if policy_type == "sb3_ppo":
            if stowage_env is None:
                raise ValueError("stowage_env required for sb3_ppo policy")
            
            self.gym_env = StowageHighLevelEnv(stowage_env)
            
            if rl_model_path and os.path.exists(rl_model_path):
                self.rl_model = PPO.load(rl_model_path, env=self.gym_env)
                print(f"Loaded PPO model from {rl_model_path}")
            else:
                # Initialize new model with your parameters
                policy_kwargs = dict(net_arch=[256, 256])
                
                self.rl_model = PPO(
                    policy="MlpPolicy",
                    env=self.gym_env,
                    learning_rate=3e-4,
                    policy_kwargs=policy_kwargs,
                    n_steps=2048,
                    batch_size=64,
                    n_epochs=10,
                    gamma=0.99,
                    gae_lambda=0.95,
                    ent_coef=0.1,
                    vf_coef=0.5,
                    clip_range=0.2,
                    verbose=1,
                    device="cpu"
                )
                print("Initialized new PPO model for high-level agent")

    def get_action(self, observation, valid_actions):
        if self.policy_type == "rule_based":
            return self._rule_based_policy(observation, valid_actions)
        elif self.policy_type == "rule_based_grouped":
            return self._rule_based_grouped_policy(observation, valid_actions)
        elif self.policy_type == "random":
            return self._random_policy(observation, valid_actions)
        elif self.policy_type == "sb3_ppo":
            return self._sb3_ppo_policy(observation, valid_actions)
        else:
            raise ValueError(f"Unknown policy type: {self.policy_type}")
    
    def _sb3_ppo_policy(self, observation, valid_actions):
        """Use PPO model to select bay"""
        if len(valid_actions) == 0:
            return None
        
        # Get valid bays
        yard_state = self._parse_yard_state(observation)
        valid_bays = set(yard_state[valid_actions, StateIds.BAY.value].astype(int))
        
        # Normalize observation
        obs_normalized = observation.astype(np.float32) / 255.0
        
        # Get action from PPO
        action, _states = self.rl_model.predict(obs_normalized, deterministic=True)
        action = int(action)
        
        # Validate against valid bays
        if action in valid_bays:
            return action
        else:
            return int(np.random.choice(list(valid_bays)))
    
    def train(self, total_timesteps=500000):
        """Train PPO model"""
        if self.rl_model is None:
            raise ValueError("RL model not initialized")
        self.rl_model.learn(total_timesteps=total_timesteps)
    
    def save(self, path):
        """Save trained model"""
        if self.rl_model is None:
            raise ValueError("No model to save")
        self.rl_model.save(path)
    
    def is_rl_based(self):
        return self.policy_type == "sb3_ppo"

    # ...rest of existing methods...

In [ ]:
#from stowage_env import StowageEnv  # Your environment
from agents.hierarchical_rule_based_agent import HierarchicalAgent

# Create your stowage environment
stowage_env = StowageEnv(...)

# Create hierarchical agent with PPO high-level agent
agent = HierarchicalAgent(
    vessel_shape=stowage_env.vessel_shape,
    yard_shape=stowage_env.yard_shape,
    num_slot_attrs=5,
    high_level_policy_type="sb3_ppo",
    low_level_policy_type="rule_based",
    stowage_env=stowage_env
)

# Train high-level PPO agent
agent.high_level_agent.train(total_timesteps=500000)

# Save model
agent.high_level_agent.save("./models/high_level_ppo")

In [ ]:

# ============================================================================
# PART 1: Define all classes in one cell
# ============================================================================

from abc import ABC, abstractmethod
from enum import Enum
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import os
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

# ============================================================================
# Enums
# ============================================================================

class StateIds(Enum):
    BAY = 0
    ROW = 1
    TIER = 2
    IS_OCCUPIED = 3
    GROUP = 4

class AgentLevel(Enum):
    HIGH_LEVEL = 0
    LOW_LEVEL = 1

# ============================================================================
# Mock Stowage Environment (for testing)
# ============================================================================

class MockStowageEnv:
    """Mock environment for testing"""
    def __init__(self, vessel_shape=(4, 3, 4), yard_shape=(4, 3, 4)):
        self.vessel_shape = vessel_shape
        self.yard_shape = yard_shape
        self.num_slot_attrs = 5
        
        self.num_vessel_bay = vessel_shape[0]//2 + vessel_shape[0]
        self.num_yard_bay = yard_shape[0]//2 + yard_shape[0]
        
        self.total_vessel_coords = self.num_vessel_bay * vessel_shape[1] * vessel_shape[2]
        self.total_yard_coords = self.num_yard_bay * yard_shape[1] * yard_shape[2]
        
        self.obs_size = (self.total_vessel_coords + self.total_yard_coords) * self.num_slot_attrs + self.num_slot_attrs
        self.step_count = 0
        self.max_steps = 100
        
    def reset(self):
        """Reset environment"""
        self.step_count = 0
        # Create mock observation
        obs = np.random.randint(0, 100, size=self.obs_size, dtype=np.int32)
        return obs
    
    def step(self, action):
        """Execute one step"""
        self.step_count += 1
        
        # Create mock next observation
        next_obs = np.random.randint(0, 100, size=self.obs_size, dtype=np.int32)
        
        # Mock reward (random or based on action)
        reward = np.random.uniform(-1, 1)
        
        # Episode done after max_steps
        done = self.step_count >= self.max_steps
        
        info = {"step": self.step_count}
        
        return next_obs, reward, done, info
    
    def get_valid_actions(self):
        """Return valid action indices (slots)"""
        # Return random subset of valid slots
        num_valid = np.random.randint(5, 20)
        valid_actions = np.random.choice(self.total_yard_coords, size=num_valid, replace=False)
        return valid_actions.tolist()

# ============================================================================
# Gymnasium Wrapper
# ============================================================================

class StowageHighLevelEnv(gym.Env):
    """Gymnasium wrapper for high-level agent (bay selection)"""
    metadata = {"render_modes": ["human"]}

    def __init__(self, stowage_env):
        super().__init__()
        self.env = stowage_env
        self.low_level_agent = None
        
        self.num_bays = stowage_env.yard_shape[0]//2 + stowage_env.yard_shape[0]
        self.action_space = spaces.Discrete(self.num_bays)
        
        num_vessel_bay = stowage_env.vessel_shape[0]//2 + stowage_env.vessel_shape[0]
        num_yard_bay = stowage_env.yard_shape[0]//2 + stowage_env.yard_shape[0]
        num_slot_attrs = 5
        
        total_vessel_coords = num_vessel_bay * stowage_env.vessel_shape[1] * stowage_env.vessel_shape[2]
        total_yard_coords = num_yard_bay * stowage_env.yard_shape[1] * stowage_env.yard_shape[2]
        
        obs_size = (total_vessel_coords + total_yard_coords) * num_slot_attrs + num_slot_attrs
        
        self.observation_space = spaces.Box(
            low=0, 
            high=255, 
            shape=(obs_size,), 
            dtype=np.float32
        )
        
        self.current_observation = None
        self.valid_actions = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_observation = self.env.reset()
        self.valid_actions = self.env.get_valid_actions()
        return self.current_observation.astype(np.float32) / 255.0, {}

    def step(self, action):
        if len(self.valid_actions) == 0:
            return self.current_observation.astype(np.float32) / 255.0, 0, True, False, {}
        
        yard_state = self._parse_yard_state(self.current_observation)
        valid_bays = set(yard_state[self.valid_actions, StateIds.BAY.value].astype(int))
        
        if int(action) not in valid_bays:
            action = int(np.random.choice(list(valid_bays)))
        
        selected_bay = int(action)
        
        if self.low_level_agent is not None:
            selected_slot = self.low_level_agent.get_action(
                self.current_observation, 
                self.valid_actions, 
                selected_bay
            )
        else:
            bay_mask = yard_state[self.valid_actions, StateIds.BAY.value] == selected_bay
            bay_valid_actions = np.array(self.valid_actions)[bay_mask]
            if len(bay_valid_actions) == 0:
                selected_slot = int(np.random.choice(self.valid_actions))
            else:
                selected_slot = int(np.random.choice(bay_valid_actions))
        
        self.current_observation, reward, done, info = self.env.step(selected_slot)
        self.valid_actions = self.env.get_valid_actions()
        obs = self.current_observation.astype(np.float32) / 255.0
        
        return obs, float(reward), done, False, info
    
    def _parse_yard_state(self, observation):
        observation = observation.astype(int)
        num_vessel_bay = self.env.vessel_shape[0]//2 + self.env.vessel_shape[0]
        num_yard_bay = self.env.yard_shape[0]//2 + self.env.yard_shape[0]
        num_slot_attrs = 5
        
        total_vessel_coords = num_vessel_bay * self.env.vessel_shape[1] * self.env.vessel_shape[2]
        total_yard_coords = num_yard_bay * self.env.yard_shape[1] * self.env.yard_shape[2]
        
        vessel_end = total_vessel_coords * num_slot_attrs
        yard_end = vessel_end + (total_yard_coords * num_slot_attrs)
        
        yard_state = observation[vessel_end:yard_end].reshape(total_yard_coords, num_slot_attrs)
        return yard_state

    def render(self):
        pass

# ============================================================================
# Base Agent
# ============================================================================

class BaseAgent(ABC):
    def __init__(self, level: AgentLevel):
        self.level = level

    @abstractmethod
    def get_action(self, observation, valid_actions):
        pass

    @abstractmethod
    def update_agent(self, reward):
        pass

# ============================================================================
# High Level Agent with PPO Support
# ============================================================================

class HighLevelAgent(BaseAgent):
    """High-Level Agent - supports rule-based and SB3 PPO"""

    def __init__(self, vessel_shape, yard_shape, num_slot_attrs, policy_type="rule_based", 
                 stowage_env=None, rl_model_path=None):
        super().__init__(AgentLevel.HIGH_LEVEL)

        self.vessel_shape = vessel_shape
        self.yard_shape = yard_shape
        self.policy_type = policy_type
        self.num_slot_attrs = num_slot_attrs
        self.episode_history = []

        self.num_vessel_bay = vessel_shape[0]//2 + vessel_shape[0]
        self.num_yard_bay = yard_shape[0]//2 + vessel_shape[0]

        self.total_vessel_coords = self.num_vessel_bay * vessel_shape[1] * vessel_shape[2]
        self.total_yard_coords = self.num_yard_bay * yard_shape[1] * yard_shape[2]

        self.rl_model = None
        self.gym_env = None
        
        if policy_type == "sb3_ppo":
            if stowage_env is None:
                raise ValueError("stowage_env required for sb3_ppo policy")
            
            self.gym_env = StowageHighLevelEnv(stowage_env)
            
            if rl_model_path and os.path.exists(rl_model_path):
                self.rl_model = PPO.load(rl_model_path, env=self.gym_env)
                print(f"✓ Loaded PPO model from {rl_model_path}")
            else:
                policy_kwargs = dict(net_arch=[256, 256])
                
                self.rl_model = PPO(
                    policy="MlpPolicy",
                    env=self.gym_env,
                    learning_rate=3e-4,
                    policy_kwargs=policy_kwargs,
                    n_steps=2048,
                    batch_size=64,
                    n_epochs=10,
                    gamma=0.99,
                    gae_lambda=0.95,
                    ent_coef=0.1,
                    vf_coef=0.5,
                    clip_range=0.2,
                    verbose=0,
                    device="cpu"
                )
                print("✓ Initialized new PPO model for high-level agent")

    def get_action(self, observation, valid_actions):
        if self.policy_type == "rule_based":
            return self._rule_based_policy(observation, valid_actions)
        elif self.policy_type == "random":
            return self._random_policy(observation, valid_actions)
        elif self.policy_type == "sb3_ppo":
            return self._sb3_ppo_policy(observation, valid_actions)
        else:
            raise ValueError(f"Unknown policy type: {self.policy_type}")
    
    def _random_policy(self, observation, valid_actions):
        """Random policy"""
        if len(valid_actions) == 0:
            return None
        
        yard_state = self._parse_yard_state(observation)
        valid_bays = set(yard_state[np.array(valid_actions), StateIds.BAY.value].astype(int))
        
        if len(valid_bays) == 0:
            return None
        
        return int(np.random.choice(list(valid_bays)))
    
    def _rule_based_policy(self, observation, valid_actions):
        """Simple rule-based policy"""
        if len(valid_actions) == 0:
            return None
        
        yard_state = self._parse_yard_state(observation)
        valid_bays = set(yard_state[np.array(valid_actions), StateIds.BAY.value].astype(int))
        
        if len(valid_bays) == 0:
            return None
        
        return int(np.random.choice(list(valid_bays)))
    
    def _sb3_ppo_policy(self, observation, valid_actions):
        """Use PPO model to select bay"""
        if len(valid_actions) == 0:
            return None
        
        yard_state = self._parse_yard_state(observation)
        valid_bays = set(yard_state[np.array(valid_actions), StateIds.BAY.value].astype(int))
        
        obs_normalized = observation.astype(np.float32) / 255.0
        action, _states = self.rl_model.predict(obs_normalized, deterministic=True)
        action = int(action)
        
        if action in valid_bays:
            return action
        else:
            return int(np.random.choice(list(valid_bays)))
    
    def train(self, total_timesteps=10000):
        """Train PPO model"""
        if self.rl_model is None:
            raise ValueError("RL model not initialized")
        print(f"Training high-level PPO agent for {total_timesteps} timesteps...")
        self.rl_model.learn(total_timesteps=total_timesteps)
        print("✓ Training complete")
    
    def save(self, path):
        """Save trained model"""
        if self.rl_model is None:
            raise ValueError("No model to save")
        self.rl_model.save(path)
        print(f"✓ Model saved to {path}")
    
    def is_rl_based(self):
        return self.policy_type == "sb3_ppo"
    
    def _parse_yard_state(self, observation):
        observation = observation.astype(int)
        vessel_end = self.total_vessel_coords * self.num_slot_attrs
        yard_end = vessel_end + (self.total_yard_coords * self.num_slot_attrs)
        yard_state = observation[vessel_end:yard_end].reshape(self.total_yard_coords, self.num_slot_attrs)
        return yard_state
    
    def update_agent(self, reward):
        self.episode_history.append({"reward": reward})

# ============================================================================
# Low Level Agent (simple rule-based for now)
# ============================================================================

class LowLevelAgent(BaseAgent):
    """Low-Level Agent - selects slot within bay"""

    def __init__(self, vessel_shape, yard_shape, num_slot_attrs, policy_type="rule_based"):
        super().__init__(AgentLevel.LOW_LEVEL)
        self.vessel_shape = vessel_shape
        self.yard_shape = yard_shape
        self.policy_type = policy_type
        self.num_slot_attrs = num_slot_attrs
        self.episode_history = []

        self.num_vessel_bay = vessel_shape[0]//2 + vessel_shape[0]
        self.num_yard_bay = yard_shape[0]//2 + yard_shape[0]

        self.total_vessel_coords = self.num_vessel_bay * vessel_shape[1] * vessel_shape[2]
        self.total_yard_coords = self.num_yard_bay * yard_shape[1] * yard_shape[2]

    def get_action(self, observation, valid_actions, selected_bay):
        if len(valid_actions) == 0:
            return None
        
        yard_state = self._parse_yard_state(observation)
        bay_mask = yard_state[np.array(valid_actions), StateIds.BAY.value] == selected_bay
        bay_valid_actions = np.array(valid_actions)[bay_mask]
        
        if len(bay_valid_actions) == 0:
            return int(np.random.choice(valid_actions))
        
        return int(np.random.choice(bay_valid_actions))
    
    def _parse_yard_state(self, observation):
        observation = observation.astype(int)
        vessel_end = self.total_vessel_coords * self.num_slot_attrs
        yard_end = vessel_end + (self.total_yard_coords * self.num_slot_attrs)
        yard_state = observation[vessel_end:yard_end].reshape(self.total_yard_coords, self.num_slot_attrs)
        return yard_state
    
    def update_agent(self, reward):
        self.episode_history.append({"reward": reward})

# ============================================================================
# Hierarchical Agent Wrapper
# ============================================================================

class HierarchicalAgent:
    """Wrapper for hierarchical agent with high-level and low-level agents"""

    def __init__(self, vessel_shape, yard_shape, num_slot_attrs,
                 high_level_policy_type="rule_based",
                 low_level_policy_type="rule_based",
                 stowage_env=None,
                 high_level_rl_path=None):
        
        self.high_level_agent = HighLevelAgent(
            vessel_shape, yard_shape, num_slot_attrs, 
            policy_type=high_level_policy_type,
            stowage_env=stowage_env,
            rl_model_path=high_level_rl_path
        )
        
        self.low_level_agent = LowLevelAgent(
            vessel_shape, yard_shape, num_slot_attrs, 
            policy_type=low_level_policy_type
        )
        
    def get_action(self, observation, valid_actions):
        if len(valid_actions) == 0:
            return None, {"error": "No valid actions"}
        
        selected_bay = self.high_level_agent.get_action(observation, valid_actions)

        if selected_bay is None:
            return valid_actions[0], {
                "selected_bay": None,
                "selected_slot": valid_actions[0],
                "agent_types": {
                    "high_level": self.high_level_agent.policy_type,
                    "low_level": self.low_level_agent.policy_type
                }
            }
        
        selected_slot = self.low_level_agent.get_action(
            observation, valid_actions, selected_bay
        )
        
        action_info = {
            "selected_bay": selected_bay,
            "selected_slot": selected_slot,
            "agent_types": {
                "high_level": self.high_level_agent.policy_type,
                "low_level": self.low_level_agent.policy_type
            }
        }

        return selected_slot, action_info

# ============================================================================
# TEST CODE
# ============================================================================

print("=" * 70)
print("TESTING HIERARCHICAL AGENT WITH SB3 PPO")
print("=" * 70)

# Create mock environment
print("\n1. Creating mock stowage environment...")
mock_env = MockStowageEnv(vessel_shape=(4, 3, 4), yard_shape=(4, 3, 4))
print(f"   ✓ Environment created: {mock_env.obs_size} observation size")

# Test 1: Rule-based agent (baseline)
print("\n2. Testing rule-based hierarchical agent...")
agent_rule = HierarchicalAgent(
    vessel_shape=mock_env.vessel_shape,
    yard_shape=mock_env.yard_shape,
    num_slot_attrs=mock_env.num_slot_attrs,
    high_level_policy_type="rule_based",
    low_level_policy_type="rule_based"
)

obs = mock_env.reset()
valid_actions = mock_env.get_valid_actions()
action, info = agent_rule.get_action(obs, valid_actions)
print(f"   ✓ Rule-based agent selected: Bay={info['selected_bay']}, Slot={info['selected_slot']}")
print(f"   ✓ Policy types: {info['agent_types']}")

# Test 2: PPO agent (with training)
print("\n3. Initializing PPO hierarchical agent...")
agent_ppo = HierarchicalAgent(
    vessel_shape=mock_env.vessel_shape,
    yard_shape=mock_env.yard_shape,
    num_slot_attrs=mock_env.num_slot_attrs,
    high_level_policy_type="sb3_ppo",
    low_level_policy_type="rule_based",
    stowage_env=mock_env
)

print("\n4. Training PPO agent (5000 timesteps)...")
agent_ppo.high_level_agent.train(total_timesteps=5000)

print("\n5. Testing trained PPO agent...")
obs = mock_env.reset()
valid_actions = mock_env.get_valid_actions()
action, info = agent_ppo.get_action(obs, valid_actions)
print(f"   ✓ PPO agent selected: Bay={info['selected_bay']}, Slot={info['selected_slot']}")
print(f"   ✓ Policy types: {info['agent_types']}")

# Test 3: Save and load model
print("\n6. Saving and loading PPO model...")
agent_ppo.high_level_agent.save("/tmp/test_ppo_model")

agent_ppo_loaded = HierarchicalAgent(
    vessel_shape=mock_env.vessel_shape,
    yard_shape=mock_env.yard_shape,
    num_slot_attrs=mock_env.num_slot_attrs,
    high_level_policy_type="sb3_ppo",
    low_level_policy_type="rule_based",
    stowage_env=mock_env,
    high_level_rl_path="/tmp/test_ppo_model"
)

obs = mock_env.reset()
valid_actions = mock_env.get_valid_actions()
action, info = agent_ppo_loaded.get_action(obs, valid_actions)
print(f"   ✓ Loaded PPO agent selected: Bay={info['selected_bay']}, Slot={info['selected_slot']}")

# Test 4: Mix of agents
print("\n7. Testing different agent combinations...")
combos = [
    ("rule_based", "rule_based"),
    ("random", "rule_based"),
    ("sb3_ppo", "rule_based"),
]

for high, low in combos:
    try:
        if high == "sb3_ppo":
            agent = HierarchicalAgent(
                vessel_shape=mock_env.vessel_shape,
                yard_shape=mock_env.yard_shape,
                num_slot_attrs=mock_env.num_slot_attrs,
                high_level_policy_type=high,
                low_level_policy_type=low,
                stowage_env=mock_env
            )
        else:
            agent = HierarchicalAgent(
                vessel_shape=mock_env.vessel_shape,
                yard_shape=mock_env.yard_shape,
                num_slot_attrs=mock_env.num_slot_attrs,
                high_level_policy_type=high,
                low_level_policy_type=low
            )
        obs = mock_env.reset()
        valid_actions = mock_env.get_valid_actions()
        action, info = agent.get_action(obs, valid_actions)
        print(f"   ✓ Combination ({high}, {low}): Bay={info['selected_bay']}, Slot={info['selected_slot']}")
    except Exception as e:
        print(f"   ✗ Combination ({high}, {low}): {str(e)}")

print("\n" + "=" * 70)
print("ALL TESTS COMPLETED SUCCESSFULLY!")
print("=" * 70)

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from agents.hierarchical_rule_based_agent import HierarchicalAgent, StateIds

# ============================================================================
# Gymnasium Wrapper for your StowageEnv
# ============================================================================

class StowageHighLevelEnv(gym.Env):
    """Wraps StowageEnv for SB3 PPO training"""
    metadata = {"render_modes": ["human"]}

    def __init__(self, stowage_env, low_level_agent):
        super().__init__()
        self.env = stowage_env
        self.low_level_agent = low_level_agent
        
        self.num_bays = stowage_env.yard_shape[0]//2 + stowage_env.yard_shape[0]
        self.action_space = spaces.Discrete(self.num_bays)
        
        num_vessel_bay = stowage_env.vessel_shape[0]//2 + stowage_env.vessel_shape[0]
        num_yard_bay = stowage_env.yard_shape[0]//2 + stowage_env.yard_shape[0]
        num_slot_attrs = 5
        
        total_vessel_coords = num_vessel_bay * stowage_env.vessel_shape[1] * stowage_env.vessel_shape[2]
        total_yard_coords = num_yard_bay * stowage_env.yard_shape[1] * stowage_env.yard_shape[2]
        
        obs_size = (total_vessel_coords + total_yard_coords) * num_slot_attrs + num_slot_attrs
        
        self.observation_space = spaces.Box(
            low=0, high=255, shape=(obs_size,), dtype=np.float32
        )
        
        self.current_observation = None
        self.valid_actions = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_observation = self.env.reset()
        self.valid_actions = self.env.get_valid_actions()
        return self.current_observation.astype(np.float32) / 255.0, {}

    def step(self, action):
        if len(self.valid_actions) == 0:
            return self.current_observation.astype(np.float32) / 255.0, 0, True, False, {}
        
        yard_state = self._parse_yard_state(self.current_observation)
        valid_bays = set(yard_state[self.valid_actions, StateIds.BAY.value].astype(int))
        
        if int(action) not in valid_bays:
            action = int(np.random.choice(list(valid_bays)))
        
        selected_bay = int(action)
        
        # Use low-level agent to select slot
        selected_slot = self.low_level_agent.get_action(
            self.current_observation, self.valid_actions, selected_bay
        )
        
        self.current_observation, reward, done, info = self.env.step(selected_slot)
        self.valid_actions = self.env.get_valid_actions()
        obs = self.current_observation.astype(np.float32) / 255.0
        
        return obs, float(reward), done, False, info
    
    def _parse_yard_state(self, observation):
        observation = observation.astype(int)
        num_vessel_bay = self.env.vessel_shape[0]//2 + self.env.vessel_shape[0]
        num_yard_bay = self.env.yard_shape[0]//2 + self.env.yard_shape[0]
        num_slot_attrs = 5
        
        total_vessel_coords = num_vessel_bay * self.env.vessel_shape[1] * self.env.vessel_shape[2]
        total_yard_coords = num_yard_bay * self.env.yard_shape[1] * self.env.yard_shape[2]
        
        vessel_end = total_vessel_coords * num_slot_attrs
        yard_end = vessel_end + (total_yard_coords * num_slot_attrs)
        
        yard_state = observation[vessel_end:yard_end].reshape(total_yard_coords, num_slot_attrs)
        return yard_state

    def render(self):
        pass

# ============================================================================
# Extended High-Level Agent with PPO Support
# ============================================================================

class HighLevelAgentWithPPO:
    """Wraps high-level policy to support both rule-based and PPO"""
    
    def __init__(self, high_level_agent, stowage_env=None, policy_type="rule_based"):
        self.high_level_agent = high_level_agent
        self.policy_type = policy_type
        self.rl_model = None
        
        if policy_type == "sb3_ppo":
            if stowage_env is None:
                raise ValueError("stowage_env required for PPO")
            
            self.gym_env = StowageHighLevelEnv(stowage_env, high_level_agent.low_level_agent)
            
            policy_kwargs = dict(net_arch=[256, 256])
            self.rl_model = PPO(
                policy="MlpPolicy",
                env=self.gym_env,
                learning_rate=3e-4,
                policy_kwargs=policy_kwargs,
                n_steps=2048,
                batch_size=64,
                n_epochs=10,
                gamma=0.99,
                gae_lambda=0.95,
                ent_coef=0.1,
                vf_coef=0.5,
                clip_range=0.2,
                verbose=0,
                device="cpu"
            )
            print("✓ Initialized PPO model for high-level agent")

    def get_action(self, observation, valid_actions):
        if self.policy_type == "sb3_ppo":
            return self._ppo_action(observation, valid_actions)
        else:
            return self.high_level_agent.get_action(observation, valid_actions)
    
    def _ppo_action(self, observation, valid_actions):
        if len(valid_actions) == 0:
            return None
        
        # Get valid bays using existing parsing logic
        yard_state = self.high_level_agent._parse_yard_state(observation)
        valid_bays = set(yard_state[np.array(valid_actions), StateIds.BAY.value].astype(int))
        
        obs_normalized = observation.astype(np.float32) / 255.0
        action, _ = self.rl_model.predict(obs_normalized, deterministic=True)
        action = int(action)
        
        if action in valid_bays:
            return action
        else:
            return int(np.random.choice(list(valid_bays)))
    
    def train(self, total_timesteps=10000):
        if self.rl_model is None:
            raise ValueError("PPO model not initialized")
        print(f"Training high-level PPO for {total_timesteps} timesteps...")
        self.rl_model.learn(total_timesteps=total_timesteps)
        print("✓ Training complete")

# ============================================================================
# TEST WITH REAL ENVIRONMENT
# ============================================================================

try:
    from envs.stack_gym import StowageEnv
    
    print("=" * 70)
    print("HIERARCHICAL AGENT TEST - RULE-BASED vs PPO")
    print("=" * 70)
    
    # Create environment
    print("\n1. Creating StowageEnv...")
    env = StowageEnv(vessel_shape=(4, 3, 4), yard_shape=(4, 3, 4), num_slot_attrs=5)
    print(f"   ✓ Vessel shape: {env.vessel_shape}, Yard shape: {env.yard_shape}")
    
    # Create rule-based agent
    print("\n2. Creating rule-based hierarchical agent...")
    agent_rule = HierarchicalAgent(
        vessel_shape=env.vessel_shape,
        yard_shape=env.yard_shape,
        num_slot_attrs=env.num_slot_attrs,
        high_level_policy_type="rule_based",
        low_level_policy_type="rule_based"
    )
    print("   ✓ Rule-based agent created")
    
    # Create PPO agent
    print("\n3. Creating PPO hierarchical agent (using your rule-based agents)...")
    agent_ppo = HierarchicalAgent(
        vessel_shape=env.vessel_shape,
        yard_shape=env.yard_shape,
        num_slot_attrs=env.num_slot_attrs,
        high_level_policy_type="rule_based",
        low_level_policy_type="rule_based"
    )
    
    # Wrap high-level agent with PPO
    ppo_wrapper = HighLevelAgentWithPPO(agent_ppo, env, policy_type="sb3_ppo")
    
    # Train PPO
    print("\n4. Training PPO agent (5000 timesteps)...")
    ppo_wrapper.train(total_timesteps=5000)
    
    # Test both agents
    print("\n5. Testing both agents (10 steps each)...")
    
    print("\n   Rule-based agent:")
    obs = env.reset()
    for step in range(10):
        valid_actions = env.get_valid_actions()
        action, info = agent_rule.get_action(obs, valid_actions)
        obs, reward, done, info = env.step(action)
        print(f"      Step {step+1}: Bay={info['selected_bay']}, Slot={info['selected_slot']}, Reward={reward:.2f}")
        if done:
            break
    
    print("\n   PPO agent (high-level) + Rule-based (low-level):")
    obs = env.reset()
    for step in range(10):
        valid_actions = env.get_valid_actions()
        ppo_bay = ppo_wrapper.get_action(obs, valid_actions)
        low_level_slot = agent_ppo.low_level_agent.get_action(obs, valid_actions, ppo_bay)
        obs, reward, done, info = env.step(low_level_slot)
        print(f"      Step {step+1}: Bay={ppo_bay}, Slot={low_level_slot}, Reward={reward:.2f}")
        if done:
            break
    
    print("\n" + "=" * 70)
    print("TEST COMPLETE!")
    print("=" * 70)

except ImportError as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from agents.hierarchical_rule_based_agent import HierarchicalAgent, StateIds
from envs.stack_gym import StackEnv

# ============================================================================
# Gymnasium Wrapper for StackEnv
# ============================================================================

class StowageHighLevelEnv(gym.Env):
    """Wraps StackEnv for SB3 PPO training"""
    metadata = {"render_modes": ["human"]}

    def __init__(self, stowage_env, low_level_agent):
        super().__init__()
        self.env = stowage_env
        self.low_level_agent = low_level_agent
        
        self.num_bays = stowage_env.num_yard_bay
        self.action_space = spaces.Discrete(self.num_bays)
        
        obs_size = stowage_env.obs_coords * stowage_env.num_slot_attrs
        
        self.observation_space = spaces.Box(
            low=0, high=255, shape=(obs_size,), dtype=np.float32
        )
        
        self.current_observation = None
        self.valid_actions = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_observation, info = self.env.reset()
        self.valid_actions = self.env._get_valid_yard_actions()
        return self.current_observation.astype(np.float32) / 255.0, {}

    def step(self, action):
        if len(self.valid_actions) == 0:
            return self.current_observation.astype(np.float32) / 255.0, 0, True, False, {}
        
        yard_state = self._parse_yard_state(self.current_observation)
        valid_bays = set(yard_state[self.valid_actions, StateIds.BAY.value].astype(int))
        
        if int(action) not in valid_bays:
            action = int(np.random.choice(list(valid_bays)))
        
        selected_bay = int(action)
        
        # Use low-level agent to select slot
        selected_slot = self.low_level_agent.get_action(
            self.current_observation, self.valid_actions, selected_bay
        )
        
        self.current_observation, reward, done, truncated, info = self.env.step(selected_slot)
        self.valid_actions = self.env._get_valid_yard_actions()
        obs = self.current_observation.astype(np.float32) / 255.0
        
        return obs, float(reward), done, truncated, info
    
    def _parse_yard_state(self, observation):
        observation = observation.astype(int)
        num_slot_attrs = 5
        
        total_vessel_coords = self.env.total_vessel_coords
        total_yard_coords = self.env.total_yard_coords
        
        vessel_end = total_vessel_coords * num_slot_attrs
        yard_end = vessel_end + (total_yard_coords * num_slot_attrs)
        
        yard_state = observation[vessel_end:yard_end].reshape(total_yard_coords, num_slot_attrs)
        return yard_state

    def render(self):
        pass

# ============================================================================
# Extended High-Level Agent with PPO Support
# ============================================================================

class HighLevelAgentWithPPO:
    """Wraps high-level policy to support both rule-based and PPO"""
    
    def __init__(self, high_level_agent, stowage_env=None, policy_type="rule_based"):
        self.high_level_agent = high_level_agent
        self.policy_type = policy_type
        self.rl_model = None
        
        if policy_type == "sb3_ppo":
            if stowage_env is None:
                raise ValueError("stowage_env required for PPO")
            
            self.gym_env = StowageHighLevelEnv(stowage_env, high_level_agent.low_level_agent)
            
            policy_kwargs = dict(net_arch=[256, 256])
            self.rl_model = PPO(
                policy="MlpPolicy",
                env=self.gym_env,
                learning_rate=3e-4,
                policy_kwargs=policy_kwargs,
                n_steps=2048,
                batch_size=64,
                n_epochs=10,
                gamma=0.99,
                gae_lambda=0.95,
                ent_coef=0.1,
                vf_coef=0.5,
                clip_range=0.2,
                verbose=0,
                device="cpu"
            )
            print("✓ Initialized PPO model for high-level agent")

    def get_action(self, observation, valid_actions):
        if self.policy_type == "sb3_ppo":
            return self._ppo_action(observation, valid_actions)
        else:
            return self.high_level_agent.get_action(observation, valid_actions)
    
    def _ppo_action(self, observation, valid_actions):
        if len(valid_actions) == 0:
            return None
        
        # Get valid bays using existing parsing logic
        yard_state = self.high_level_agent._parse_yard_state(observation)
        valid_bays = set(yard_state[np.array(valid_actions), StateIds.BAY.value].astype(int))
        
        obs_normalized = observation.astype(np.float32) / 255.0
        action, _ = self.rl_model.predict(obs_normalized, deterministic=True)
        action = int(action)
        
        if action in valid_bays:
            return action
        else:
            return int(np.random.choice(list(valid_bays)))
    
    def train(self, total_timesteps=10000):
        if self.rl_model is None:
            raise ValueError("PPO model not initialized")
        print(f"Training high-level PPO for {total_timesteps} timesteps...")
        self.rl_model.learn(total_timesteps=total_timesteps)
        print("✓ Training complete")

# ============================================================================
# TEST WITH REAL STACK ENVIRONMENT
# ============================================================================

try:
    print("=" * 70)
    print("HIERARCHICAL AGENT TEST - RULE-BASED vs PPO (StackEnv)")
    print("=" * 70)
    
    # Create environment with StackEnv
    print("\n1. Creating StackEnv...")
    config = {
        "vessel_shape": (4, 3, 4),
        "yard_shape": (4, 3, 4),
        "num_containers": 10,
        "group_num": 2,
        "seed": 42
    }
    env = StackEnv(config=config)
    print(f"   ✓ StackEnv created")
    print(f"   ✓ Vessel shape: {env.vessel_shape}, Yard shape: {env.yard_shape}")
    print(f"   ✓ Num containers: {env.num_containers}, Num groups: {env.group_num}")
    
    # Create rule-based agent
    print("\n2. Creating rule-based hierarchical agent...")
    agent_rule = HierarchicalAgent(
        vessel_shape=env.vessel_shape,
        yard_shape=env.yard_shape,
        num_slot_attrs=env.num_slot_attrs,
        high_level_policy_type="rule_based",
        low_level_policy_type="rule_based"
    )
    print("   ✓ Rule-based agent created")
    
    # Create PPO agent
    print("\n3. Creating PPO hierarchical agent (using your rule-based agents)...")
    agent_ppo = HierarchicalAgent(
        vessel_shape=env.vessel_shape,
        yard_shape=env.yard_shape,
        num_slot_attrs=env.num_slot_attrs,
        high_level_policy_type="rule_based",
        low_level_policy_type="rule_based"
    )
    
    # Wrap high-level agent with PPO
    ppo_wrapper = HighLevelAgentWithPPO(agent_ppo, env, policy_type="sb3_ppo")
    
    # Train PPO
    print("\n4. Training PPO agent (5000 timesteps)...")
    ppo_wrapper.train(total_timesteps=5000)
    
    # Test both agents
    print("\n5. Testing both agents (10 steps each)...")
    
    print("\n   Rule-based agent:")
    obs, info = env.reset()
    for step in range(10):
        valid_actions = env._get_valid_yard_actions()
        if len(valid_actions) == 0:
            print(f"      No more valid actions")
            break
        action, action_info = agent_rule.get_action(obs, valid_actions)
        obs, reward, done, truncated, env_info = env.step(action)
        print(f"      Step {step+1}: Bay={action_info['selected_bay']}, Slot={action_info['selected_slot']}, Reward={reward:.2f}")
        if done:
            break
    
    print("\n   PPO agent (high-level) + Rule-based (low-level):")
    obs, info = env.reset()
    for step in range(10):
        valid_actions = env._get_valid_yard_actions()
        if len(valid_actions) == 0:
            print(f"      No more valid actions")
            break
        ppo_bay = ppo_wrapper.get_action(obs, valid_actions)
        if ppo_bay is None:
            print(f"      PPO returned None bay")
            break
        low_level_slot = agent_ppo.low_level_agent.get_action(obs, valid_actions, ppo_bay)
        obs, reward, done, truncated, env_info = env.step(low_level_slot)
        print(f"      Step {step+1}: Bay={ppo_bay}, Slot={low_level_slot}, Reward={reward:.2f}")
        if done:
            break
    
    print("\n" + "=" * 70)
    print("TEST COMPLETE!")
    print("=" * 70)

except ImportError as e:
    print(f"Import Error: {e}")
    import traceback
    traceback.print_exc()
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
env = StackEnv(config=config)

In [ ]:
env.observation_space

In [ ]:
env.action_space

In [ ]:
len(env.observation_space.high)

In [ ]:
env.action_space.n

In [ ]:
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent, StateIds
from envs.stack_gym import StackEnv

In [ ]:
config = {
        "vessel_shape": (4, 3, 4),
        "yard_shape": (4, 3, 4),
        "num_containers": 10,
        "group_num": 2,
        "seed": 42
    }
env = StackEnv(config=config)
agent = HierarchicalAgent(
    vessel_shape=config["vessel_shape"],
    yard_shape=config["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based",
    low_level_policy_type="rule_based"
) 
env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)

In [ ]:
env_high.observation_space

In [ ]:
env_high.action_space.n

In [ ]:
import numpy as np
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent, StateIds
from envs.stack_gym import StackEnv

# Setup
config = {
    "vessel_shape": (5, 5, 4),
    "yard_shape": (5, 5, 4),
    "num_containers": 100,
    "group_num": 7,
    "group_placement": "random",
    "seed": 4307,
}

env = StackEnv(config=config)
agent = HierarchicalAgent(
    vessel_shape=config["vessel_shape"],
    yard_shape=config["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based_grouped",
    low_level_policy_type="rule_based_grouped"
)
env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)

# Test with random high-level actions
print("=" * 70)
print("Testing StackHighLevelEnv with Random High-Level Actions")
print("=" * 70)

obs, info = env_high.reset()
print(f"\nObservation shape: {obs.shape}")
print(f"Action space: {env_high.action_space}")
print(f"Num bays (high-level actions): {env_high.num_bays}")

total_reward = 0.0

for step in range(100):
    # Random high-level action (bay selection)
    act_index = env_high.action_space.sample()
    
    action = env_high.baynum_list[act_index-1]


    
    print(f"\nStep {step + 1}:")
    print(f"  High-level action (bay): {action}")
    
    # Execute step
    obs, reward, done, truncated, info = env_high.step(action)
    total_reward += reward
    
    print(f"  Reward: {reward:.4f}")
    print(f"  Done: {done}, Truncated: {truncated}")
    
    if done or truncated:
        print("\nEpisode finished!")
        break

print(f"\nTotal Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent, StateIds
from envs.stack_gym import StackEnv

# Configuration
config_dict = {
    "vessel_shape": (5, 5, 4),
    "yard_shape": (5, 5, 4),
    "num_containers": 100,
    "group_num": 7,
    "group_placement": "random",
    "seed": 4307,
}

# Parameters
MAX_STEPS = 100

# Create environment and agent
env = StackEnv(config=config_dict, render_mode="rgb_array")
agent = HierarchicalAgent(
    vessel_shape=config_dict["vessel_shape"],
    yard_shape=config_dict["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based_grouped",
    low_level_policy_type="rule_based_grouped"
)

env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)

# Reset environment
observation, info = env_high.reset()
total_reward = 0.0
step_count = 0

print("="*60)
print(f"Testing StackHighLevelEnv (Max {MAX_STEPS} steps)")
print("="*60)

# Main loop
for step in range(1, MAX_STEPS + 1):
    print(f"\nSTEP {step}")
    print("-"*60)
    
    # Random high-level action (bay selection)
    act_index = env_high.action_space.sample()
    
    action = env_high.baynum_list[act_index-1]
    print(f"Random bay action: {action}")
    
    # Execute step
    observation, reward, done, truncated, step_info = env_high.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Reward: {reward:.2f}")
    print(f"Total Reward: {total_reward:.2f}")
    
    # Render
    img = env_high.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Bay: {action} | Reward: {reward:.2f} | Total: {total_reward:.2f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    
    if done or truncated:
        print("\n" + "="*60)
        print("EPISODE FINISHED!")
        print("="*60)
        break

print("\n" + "="*60)
print(f"Summary: {step_count} steps, Total Reward: {total_reward:.2f}")
print("="*60)

In [ ]:
# Train PPO on StackHighLevelEnv

import numpy as np
from stable_baselines3 import PPO
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

# Configuration
config_dict = {
    "vessel_shape": (5, 5, 4),
    "yard_shape": (5, 5, 4),
    "num_containers": 100,
    "group_num": 7,
    "group_placement": "random",
    "seed": 4307,
}

# Create environment and agent
env = StackEnv(config=config_dict)
agent = HierarchicalAgent(
    vessel_shape=config_dict["vessel_shape"],
    yard_shape=config_dict["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based",
    low_level_policy_type="rule_based_grouped"
)

env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)

# Initialize PPO
print("=" * 70)
print("TRAINING PPO ON STACKHIGHLEVELENV")
print("=" * 70)

policy_kwargs = dict(net_arch=[256, 256])
model = PPO(
    policy="MlpPolicy",
    env=env_high,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 50000 timesteps...")
model.learn(total_timesteps=50_000)
print("✓ Training complete")

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env_high.reset()
total_reward = 0.0

for step in range(20):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env_high.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

# Save model
model.save("ppo_stowage_high_level")
print("✓ Model saved to ppo_stowage_high_level")

Test flat RL without action masking

In [ ]:
# Train PPO on StackHighLevelEnv

import numpy as np
from stable_baselines3 import PPO
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy

from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback



# Configuration
config_dict = {
    "vessel_shape": (3, 2, 2),
    "yard_shape": (3, 2, 2),
    "num_containers": 12,
    "group_num": 3,
    "group_placement": "random",
    "seed": 4307,
    "reward_scheme": "simple_single_stack",
    "observation_type": "stack_features"
}

# Create environment and agent
env = StackEnv(config=config_dict,  render_mode="rgb_array")
"""
agent = HierarchicalAgent(
    vessel_shape=config_dict["vessel_shape"],
    yard_shape=config_dict["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based",
    low_level_policy_type="rule_based_grouped"
)

def mask_fn(env: gym.Env) -> np.ndarray:
        return env._get_action_mask()
"""
"""
env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)
env_high_mask = ActionMasker(env_high, mask_fn)
"""

# Initialize PPO
print("=" * 70)
print("TRAINING PPO ON STACKHIGHLEVELENV")
print("=" * 70)

policy_kwargs = dict(net_arch=[256, 256, 32])
model = PPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 100000 timesteps...")
#model.learn(total_timesteps=50_000, callback = callback)
model.learn(total_timesteps=100_000)
print("Training complete")

# Test
"""
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env_high.reset()
total_reward = 0.0

for step in range(100):
    action_masks = get_action_masks(env_high_mask)

    action, _states = model.predict(obs, deterministic=False,action_masks=action_masks)
    obs, reward, done, truncated, info = env_high.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)
"""

# Save model
model.save("ppo_stack_flat")
print("Model saved to ppo_stack_flat")

In [ ]:
from sb3_contrib.common.maskable.utils import get_action_masks

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0

for step in range(12):

    action, _states = model.predict(obs)
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(12):
    action, _states = model.predict(obs)
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

Test flat RL with action masking

Flat RL with new obs space on simple reward scheme

In [ ]:
# Train PPO on StackHighLevelEnv

import numpy as np
from stable_baselines3 import PPO
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy

from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback



# Configuration
config_dict = {
    "vessel_shape": (3, 3, 3),
    "yard_shape": (3, 3, 3),
    "num_containers": 27,
    "group_num": 3,
    "group_placement": "random",
    "seed": 4307,
    "reward_scheme": "simple_single_stack",
    #"observation_type": "stack_features"
}

# Create environment and agent
env = StackEnv(config=config_dict,  render_mode="rgb_array")



policy_kwargs = dict(net_arch=[256, 256, 32])
model = MaskablePPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 1500000 timesteps")
#model.learn(total_timesteps=500_000, callback = callback)
model.learn(total_timesteps=10_000)
print("Training complete")


# Save model
#model.save("ppo_stack_flat_masked_1.5M_simple_reward")
#print("Model saved to ppo_stack_flat_masked_1.5M_simple_reward")

In [ ]:
from sb3_contrib.common.maskable.utils import get_action_masks

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0

for step in range(100):

    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(64):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    action_masks = get_action_masks(env)
    #print(f"Action mask: {action_masks}")
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(64):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(60):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(60):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
# Train PPO on StackHighLevelEnv

import numpy as np
from stable_baselines3 import PPO
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy

from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback



# Configuration
config_dict = {
    "vessel_shape": (3, 3, 3),
    "yard_shape": (3, 3, 3),
    "num_containers": 27,
    "group_num": 3,
    "group_placement": "random",
    "seed": 4307,
    #"reward_scheme": "simple_single_stack",
    "observation_type": "stack_features"
}

# Create environment and agent
env = StackEnv(config=config_dict,  render_mode="rgb_array")



policy_kwargs = dict(net_arch=[256, 256, 32])
model = MaskablePPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 1500000 timesteps")
#model.learn(total_timesteps=500_000, callback = callback)
model.learn(total_timesteps=1_500_000)
print("Training complete")


# Save model
model.save("ppo_stack_flat_masked_1.5M_complex_reward")
print("Model saved to ppo_stack_flat_masked_1.5M_complex_reward")

In [ ]:
from sb3_contrib.common.maskable.utils import get_action_masks

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0

for step in range(60):

    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(60):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset(seed=5)
total_reward = 0.0
step_count = 0

for step in range(60):
    action, _states = model.predict(obs,deterministic=True, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

Flat RL solution with action masking for 5 bay 5 groups

In [ ]:
# Train PPO on StackHighLevelEnv

import numpy as np
from stable_baselines3 import PPO
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy

from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback



# Configuration
config_dict = {
    "vessel_shape": (5, 4, 4),
    "yard_shape": (5, 4, 4),
    "num_containers": 80,
    "group_num": 5,
    "group_placement": "random",
    "seed": 4307,
    #"reward_scheme": "simple_single_stack",
    "observation_type": "stack_features"
}

# Create environment and agent
env = StackEnv(config=config_dict,  render_mode="rgb_array")



policy_kwargs = dict(net_arch=[256, 256, 32])
model = MaskablePPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 1500000 timesteps")
#model.learn(total_timesteps=500_000, callback = callback)
model.learn(total_timesteps=1_500_000)
print("Training complete")


# Save model
model.save("ppo_stack_flat_masked_1.5M_complex_reward_bigenv")
print("Model saved to ppo_stack_flat_masked_1.5M_complex_reward_bigenv")

In [ ]:
from sb3_contrib.common.maskable.utils import get_action_masks

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0

for step in range(60):

    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset(seed=5)
total_reward = 0.0
step_count = 0

for step in range(60):
    action, _states = model.predict(obs,deterministic=True, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset(seed=5)
total_reward = 0.0
step_count = 0

for step in range(60):
    action, _states = model.predict(obs,deterministic=True, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset(seed=5)
total_reward = 0.0
step_count = 0

for step in range(60):
    action, _states = model.predict(obs,deterministic=True, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(12):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(12):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv


print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env.reset()
total_reward = 0.0
step_count = 0

for step in range(12):
    action, _states = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
# Train PPO on StackHighLevelEnv

import numpy as np
from stable_baselines3 import PPO
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy

from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
import gymnasium as gym

from stable_baselines3.common.callbacks import BaseCallback

"""
class RewardLoggingCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.current_episode_reward = 0.0

    def _on_step(self) -> bool:
        # Accumulate rewards
        self.current_episode_reward += self.locals["rewards"][0]
        
        # Check if episode is done
        if self.locals["dones"][0]:
            self.episode_rewards.append(self.current_episode_reward)
            self.logger.record("episode/reward", self.current_episode_reward)
            self.current_episode_reward = 0.0
        
        return True


callback = RewardLoggingCallback()

model.learn(total_timesteps=50_000, callback=callback)
print("✓ Training complete")
"""

# Configuration
config_dict = {
    "vessel_shape": (5, 5, 4),
    "yard_shape": (5, 5, 4),
    "num_containers": 100,
    "group_num": 7,
    "group_placement": "random",
    "seed": 4307,
}

# Create environment and agent
env = StackEnv(config=config_dict)
agent = HierarchicalAgent(
    vessel_shape=config_dict["vessel_shape"],
    yard_shape=config_dict["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based",
    low_level_policy_type="rule_based_grouped"
)

def mask_fn(env: gym.Env) -> np.ndarray:
        return env._get_action_mask()

env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)
env_high_mask = ActionMasker(env_high, mask_fn)

# Initialize PPO
print("=" * 70)
print("TRAINING PPO ON STACKHIGHLEVELENV")
print("=" * 70)

policy_kwargs = dict(net_arch=[256, 256, 32])
model = MaskablePPO(
    policy="MlpPolicy",
    env=env_high_mask,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 50000 timesteps...")
#model.learn(total_timesteps=50_000, callback = callback)
model.learn(total_timesteps=50_000)
print("✓ Training complete")

# Test
"""
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env_high.reset()
total_reward = 0.0

for step in range(100):
    action_masks = get_action_masks(env_high_mask)

    action, _states = model.predict(obs, deterministic=False,action_masks=action_masks)
    obs, reward, done, truncated, info = env_high.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)
"""

# Save model
model.save("ppo_stowage_high_level")
print("✓ Model saved to ppo_stowage_high_level")

In [ ]:
obs, info = env_high.reset()
print(env_high._get_action_mask())

In [ ]:
from sb3_contrib.common.maskable.utils import get_action_masks

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MODEL")
print("=" * 70)

obs, info = env_high.reset()
total_reward = 0.0

for step in range(100):

    action, _states = model.predict(obs, deterministic=False, action_masks=env_high._get_action_mask())
    obs, reward, done, truncated, info = env_high.step(action)
    total_reward += reward
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

# Save model
model.save("ppo_stowage_high_level")
print("✓ Model saved to ppo_stowage_high_level")

In [ ]:
import matplotlib.pyplot as plt
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

# Configuration
config_dict = {
    "vessel_shape": (5, 5, 4),
    "yard_shape": (5, 5, 4),
    "num_containers": 100,
    "group_num": 7,
    "group_placement": "random",
    "seed": 4307,
}

# Create environment with render_mode
env = StackEnv(config=config_dict, render_mode="rgb_array")
agent = HierarchicalAgent(
    vessel_shape=config_dict["vessel_shape"],
    yard_shape=config_dict["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based",
    low_level_policy_type="rule_based_grouped"
)

env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)

print("=" * 70)
print("TESTING TRAINED MODEL WITH VISUALIZATION")
print("=" * 70)

obs, info = env_high.reset()
total_reward = 0.0
step_count = 0

for step in range(110):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env_high.step(action)
    total_reward += reward
    step_count += 1
    
    print(f"Step {step_count}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}")
    
    # Render visualization
    img = env_high.render()
    if img is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Step {step_count} | Action: {action} | Reward: {reward:.4f} | Total: {total_reward:.4f}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"  Warning: No image rendered at step {step_count}")
    
    if done or truncated:
        print("\n" + "=" * 70)
        print("EPISODE FINISHED!")
        print("=" * 70)
        break

print(f"\nFinal Summary: {step_count} steps, Total Reward: {total_reward:.4f}")
print("=" * 70)

In [ ]:
from sb3_contrib import MaskablePPO
# Train MaskablePPO on StackHighLevelEnv

import numpy as np
from envs.hierarchical_envs.hierarchical_sb3 import StackHighLevelEnv
from agents.hierarchical_rule_based_agent import HierarchicalAgent
from envs.stack_gym import StackEnv

# Configuration
config_dict = {
    "vessel_shape": (5, 5, 4),
    "yard_shape": (5, 5, 4),
    "num_containers": 100,
    "group_num": 7,
    "group_placement": "random",
    "seed": 4307,
}

# Create environment and agent
env = StackEnv(config=config_dict)
agent = HierarchicalAgent(
    vessel_shape=config_dict["vessel_shape"],
    yard_shape=config_dict["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based",
    low_level_policy_type="rule_based_grouped"
)

env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)

# Define mask function for MaskablePPO
def mask_fn(env):
    """Returns action mask from environment"""
    return env.observation_space["action_mask"]

# Initialize MaskablePPO
print("=" * 70)
print("TRAINING MASKABLEPPO ON STACKHIGHLEVELENV")
print("=" * 70)

policy_kwargs = dict(net_arch=[256, 256])
model = MaskablePPO(
    policy="MultiInputPolicy",
    env=env_high,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

# Train
print("\nTraining for 500000 timesteps...")
print("Invalid actions masked with -inf in logits...")
model.learn(total_timesteps=500_000)
print("✓ Training complete")

# Test
print("\n" + "=" * 70)
print("TESTING TRAINED MASKABLEPPO MODEL")
print("=" * 70)

obs, info = env_high.reset()
total_reward = 0.0

for step in range(20):
    # MaskablePPO automatically applies mask
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env_high.step(action)
    total_reward += reward
    
    action_mask = obs["action_mask"]
    valid_actions_count = np.sum(action_mask)
    
    print(f"Step {step + 1}: Action={action}, Reward={reward:.4f}, Total={total_reward:.4f}, Valid={int(valid_actions_count)}")
    
    if done or truncated:
        print("Episode finished!")
        break

print(f"\nFinal Total Reward: {total_reward:.4f}")
print("=" * 70)

# Save model
model.save("maskableppo_stowage_high_level")
print("✓ Model saved to maskableppo_stowage_high_level")

In [ ]:
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
import numpy as np

# Configuration
config_dict = {
    "vessel_shape": (5, 5, 4),
    "yard_shape": (5, 5, 4),
    "num_containers": 100,
    "group_num": 7,
    "group_placement": "random",
    "seed": 4307,
}

# Create environment and agent
env = StackEnv(config=config_dict)
agent = HierarchicalAgent(
    vessel_shape=config_dict["vessel_shape"],
    yard_shape=config_dict["yard_shape"],
    num_slot_attrs=5,
    high_level_policy_type="rule_based",
    low_level_policy_type="rule_based_grouped"
)

env_high = StackHighLevelEnv(env, low_level_agent=agent.low_level_agent)

# Define mask function
def mask_fn(env):
    """Extract action mask from environment's observation"""
    return env.observation_space["action_mask"].sample() if hasattr(env, 'observation_space') else np.ones(env.num_bays, dtype=np.uint8)

# Wrap with ActionMasker
masked_env = ActionMasker(env_high, mask_fn)

print("=" * 70)
print("TRAINING MASKABLEPPO ON STACKHIGHLEVELENV")
print("=" * 70)

policy_kwargs = dict(net_arch=[256, 256])
model = MaskablePPO(
    policy="MultiInputPolicy",
    env=masked_env,
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.3,
    vf_coef=0.5,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

print("\nTraining for 500000 timesteps...")
print("Invalid actions masked with -inf in logits...")
model.learn(total_timesteps=500_000)
print("✓ Training complete")